<a href="https://colab.research.google.com/github/afullhart/climateanalogs/blob/main/Colab/Accuracy_Score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
%reset -f

In [4]:
!pip install rioxarray

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.4/72.4 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 26.0 MB/s eta 0:00:00
  Attempting uninstall: xarray
    Found existing installation: xarray 2025.12.0
    Uninstalling xarray-2025.12.0:
      Successfully uninstalled xarray-2025.12.0


In [2]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [17]:
import numpy as np
import pandas as pd
import rasterio
from rasterio.features import rasterize
import geopandas as gpd

# 1. Define Paths (Using your established working directories)
iso_raster_path = '/content/drive/My Drive/Colab Notebooks/Analogs/IsoCluster.tif'
elc_shapefile_path = '/content/drive/My Drive/Colab Notebooks/Analogs/us_eco_l3_clip/us_eco_l3_clip.shp'
zonalf_path = '/content/drive/My Drive/Colab Notebooks/Analogs/Zonal/Zonal_tavg.csv'

# 2. Safely map Original Cluster ID to Temperature Order using the Zonal CSV
raw_df = pd.read_csv(zonalf_path)
sorted_indices = raw_df['MEAN'].sort_values(ascending=False).index
temp_to_orig_map = {rank + 1: original_idx + 1 for rank, original_idx in enumerate(sorted_indices)}

# 3. Load the ISODATA clusters using native rasterio
with rasterio.open(iso_raster_path) as src:
    iso_arr = src.read(1)
    iso_transform = src.transform

# 4. Map the ER3 codes (US_L3CODE) to your 5 Diagnostic Groups
er3_to_macro = {
    14: 1, 81: 1,
    13: 2, 20: 2, 22: 2, 24: 2,
    5: 3, 18: 3, 19: 3, 21: 3, 80: 3,
    23: 4, 79: 4,
    25: 5, 26: 5
}

elc_gdf = gpd.read_file(elc_shapefile_path)
elc_gdf['Macro_ID'] = elc_gdf['US_L3CODE'].astype(int).map(er3_to_macro)
elc_gdf = elc_gdf.dropna(subset=['Macro_ID'])

# 5. Rasterize the ELC polygons to perfectly align with the ISODATA grid
elc_arr = rasterize(
    shapes=((geom, int(value)) for geom, value in zip(elc_gdf.geometry, elc_gdf['Macro_ID'])),
    out_shape=iso_arr.shape,
    transform=iso_transform,
    fill=0,
    dtype='int16'
)

# 6. Mask and flatten to only evaluate valid pixels within the study area
valid_mask = (iso_arr >= 1) & (iso_arr <= 15) & (elc_arr > 0)
iso_flat = iso_arr[valid_mask]
elc_flat = elc_arr[valid_mask]

# 7. Generate Spatial Overlap Matrix (In-Memory Contingency Table)
overlap_matrix = pd.crosstab(iso_flat, elc_flat)
total_valid_pixels = len(iso_flat)

# 8. Define Case-Normalized Groups (Temperature Ordered)
cn_groups = {
    'Hot Desert': {'macro_id': 1, 'temp_clusters': [1, 2, 3]},
    'Great Basin/Colorado Plateau': {'macro_id': 2, 'temp_clusters': [6, 7, 8, 9, 10, 11, 12, 14]},
    'Northern Basin/Mountain': {'macro_id': 3, 'temp_clusters': [12, 13, 14, 15]},
    'Mogollon/Madrean': {'macro_id': 4, 'temp_clusters': [3, 5, 13]},
    'Great Plains': {'macro_id': 5, 'temp_clusters': [4, 11]}
}

# 9. Calculate Metrics Dynamically to construct Table 3
results = []
for group_name, params in cn_groups.items():
    target_macro_id = params['macro_id']
    temp_clusters = params['temp_clusters']

    # Translate temperature orders back to original raster IDs for matrix indexing
    orig_clusters = [temp_to_orig_map[t] for t in temp_clusters]

    if target_macro_id not in overlap_matrix.columns:
        continue

    # Extract Base Area Footprints
    elc_area = overlap_matrix[target_macro_id].sum()
    outside_area = total_valid_pixels - elc_area
    cluster_area = overlap_matrix.loc[overlap_matrix.index.isin(orig_clusters)].sum().sum()

    # Derive the Confusion Matrix
    true_positives = overlap_matrix.loc[overlap_matrix.index.isin(orig_clusters), target_macro_id].sum()
    false_positives = cluster_area - true_positives
    false_negatives = elc_area - true_positives
    true_negatives = outside_area - false_positives

    # Compute Final Evaluation Metrics
    tpr = true_positives / elc_area if elc_area > 0 else 0
    tnr = true_negatives / outside_area if outside_area > 0 else 0
    case_normalized_rate = (tpr + tnr) / 2
    accuracy = (true_positives + true_negatives) / total_valid_pixels

    results.append({
        'Diagnostic Group': group_name,
        'Macro-clusters': str(temp_clusters),
        'TP': true_positives,
        'TN': true_negatives,
        'FP': false_positives,
        'FN': false_negatives,
        'Prediction Accuracy': round(accuracy, 4),
        'Case-Normalized Rate': round(case_normalized_rate, 4)
    })

# Output the final metrics exactly like Table 3
metrics_df = pd.DataFrame(results)

print("Table 3: ERA3 Diagnostic Group Accuracy Scores (Case-Normalized Optimization)")
print("="*95)
print(metrics_df.to_string(index=False))

Table 3: ERA3 Diagnostic Group Accuracy Scores (Case-Normalized Optimization)
            Diagnostic Group               Macro-clusters     TP      TN     FP     FN  Prediction Accuracy  Case-Normalized Rate
                  Hot Desert                    [1, 2, 3] 193286 1320108 102075   7969               0.9322                0.9443
Great Basin/Colorado Plateau [6, 7, 8, 9, 10, 11, 12, 14] 773312  592777 138818 118531               0.8415                0.8387
     Northern Basin/Mountain             [12, 13, 14, 15] 161982 1133317 316935  11204               0.7979                0.8584
            Mogollon/Madrean                   [3, 5, 13] 183855 1282208 131012  26363               0.9031                0.8909
                Great Plains                      [4, 11] 135118 1407027  69475  11818               0.9499                0.9363


In [19]:
import numpy as np
import pandas as pd
import rasterio
from rasterio.features import rasterize
import geopandas as gpd

# 1. Define Paths
iso_raster_path = '/content/drive/My Drive/Colab Notebooks/Analogs/IsoCluster.tif'
mlra_shapefile_path = '/content/drive/My Drive/Colab Notebooks/Analogs/mlra_clip/mlra_clip.shp'
zonalf_path = '/content/drive/My Drive/Colab Notebooks/Analogs/Zonal/Zonal_tavg.csv'

# 2. Map Original Cluster ID to Temperature Order
raw_df = pd.read_csv(zonalf_path)
sorted_indices = raw_df['MEAN'].sort_values(ascending=False).index
orig_to_temp_map = {original_idx + 1: rank + 1 for rank, original_idx in enumerate(sorted_indices)}

# 3. Load the ISODATA clusters
with rasterio.open(iso_raster_path) as src:
    iso_arr = src.read(1)
    iso_transform = src.transform

# 4. Map the MLRARSYM codes to the 6 Diagnostic Groups
mlra_groups = {
    1: ['30', '40'],                                                               # Hot Desert
    2: ['24', '27', '28A', '29', '34A', '34B', '35', '42B'],                       # Great Basin/Colorado Plateau
    3: ['11', '13', '22A', '23', '25', '26', '28B', '36', '46', '47', '48A', '51'],# Northern Basin/Mountain
    4: ['38', '41'],                                                               # Mogollon/Madrean
    5: ['39'],                                                                     # Northern Transition
    6: ['42A', '42C', '70A', '70B', '77B', '77C', '77D', '77E']                    # Great Plains
}

mlra_to_macro = {code: macro_id for macro_id, codes in mlra_groups.items() for code in codes}
macro_names = {
    1: "Hot Desert",
    2: "Great Basin/Colorado Plateau",
    3: "Northern Basin/Mountain",
    4: "Mogollon/Madrean",
    5: "Northern Transition",
    6: "Great Plains"
}

# Load MLRA shapefile and map codes
mlra_gdf = gpd.read_file(mlra_shapefile_path)
mlra_gdf['Macro_ID'] = mlra_gdf['MLRARSYM'].map(mlra_to_macro)
mlra_gdf = mlra_gdf.dropna(subset=['Macro_ID'])

# 5. Rasterize the MLRA polygons to perfectly align with the ISODATA grid
mlra_arr = rasterize(
    shapes=((geom, int(value)) for geom, value in zip(mlra_gdf.geometry, mlra_gdf['Macro_ID'])),
    out_shape=iso_arr.shape,
    transform=iso_transform,
    fill=0,
    dtype='int16'
)

# 6. Mask and flatten to only evaluate valid pixels within the study area
valid_mask = (iso_arr >= 1) & (iso_arr <= 15) & (mlra_arr > 0)
iso_flat = iso_arr[valid_mask]
mlra_flat = mlra_arr[valid_mask]

# 7. Generate Spatial Overlap Matrix and Pre-calculate areas
overlap_matrix = pd.crosstab(iso_flat, mlra_flat)
total_valid_pixels = len(iso_flat)
cluster_areas = pd.Series(iso_flat).value_counts()
mlra_areas = pd.Series(mlra_flat).value_counts()

# 8. Optimize & Calculate Metrics Dynamically for MLRAs
results = []
for macro_id, group_name in macro_names.items():
    if macro_id not in overlap_matrix.columns:
        continue

    elc_total_area = mlra_areas[macro_id]
    outside_total_area = total_valid_pixels - elc_total_area

    assigned_for_normalized = []

    # --- Case-Normalized Optimization Loop ---
    for cluster_id in overlap_matrix.index:
        overlap_area = overlap_matrix.loc[cluster_id, macro_id] if macro_id in overlap_matrix.columns else 0
        cluster_total_area = cluster_areas[cluster_id]
        outside_spill = cluster_total_area - overlap_area

        tpr_gain = overlap_area / elc_total_area
        tnr_loss = outside_spill / outside_total_area

        # Assign if the True Positive Rate gain exceeds the True Negative Rate loss
        if tpr_gain > tnr_loss:
            assigned_for_normalized.append(cluster_id)

    # Calculate final metrics for the optimized MLRA group
    temp_clusters = sorted([orig_to_temp_map[c] for c in assigned_for_normalized])

    elc_area = elc_total_area
    outside_area = outside_total_area
    cluster_area = cluster_areas[cluster_areas.index.isin(assigned_for_normalized)].sum() if len(assigned_for_normalized) > 0 else 0

    true_positives = overlap_matrix.loc[overlap_matrix.index.isin(assigned_for_normalized), macro_id].sum() if len(assigned_for_normalized) > 0 else 0
    false_positives = cluster_area - true_positives
    false_negatives = elc_area - true_positives
    true_negatives = outside_area - false_positives

    tpr = true_positives / elc_area if elc_area > 0 else 0
    tnr = true_negatives / outside_area if outside_area > 0 else 0
    case_normalized_rate = (tpr + tnr) / 2
    accuracy = (true_positives + true_negatives) / total_valid_pixels

    results.append({
        'Diagnostic Group': group_name,
        'Macro-clusters': str(temp_clusters),
        'TP': true_positives,
        'TN': true_negatives,
        'FP': false_positives,
        'FN': false_negatives,
        'Prediction Accuracy': round(accuracy, 4),
        'Case-Normalized Rate': round(case_normalized_rate, 4)
    })

# Output the final MLRA metrics
metrics_df = pd.DataFrame(results)

print("Table 3 Extension: MLRA Diagnostic Group Accuracy Scores (Case-Normalized Optimization)")
print("="*95)
print(metrics_df.to_string(index=False))

Table 3 Extension: MLRA Diagnostic Group Accuracy Scores (Case-Normalized Optimization)
            Diagnostic Group           Macro-clusters     TP      TN     FP     FN  Prediction Accuracy  Case-Normalized Rate
                  Hot Desert                [1, 2, 3] 175062 1320489 120299   7588               0.9212                0.9375
Great Basin/Colorado Plateau [6, 7, 8, 9, 10, 11, 12] 558116  750999 202087 112236               0.8064                0.8103
     Northern Basin/Mountain     [10, 12, 13, 14, 15] 317140  939908 334054  32336               0.7743                0.8226
            Mogollon/Madrean               [3, 5, 13] 132546 1295971 182321  12600               0.8799                0.8949
         Northern Transition              [5, 12, 13]  66553 1264568 280844  11473               0.8199                0.8356
                Great Plains               [4, 5, 11] 179632 1297759 127891  18156               0.9100                0.9092


In [20]:
import numpy as np
import pandas as pd
import rasterio
from rasterio.features import rasterize
import geopandas as gpd

# 1. Define Paths
iso_raster_path = '/content/drive/My Drive/Colab Notebooks/Analogs/IsoCluster.tif'
elc_shapefile_path = '/content/drive/My Drive/Colab Notebooks/Analogs/us_eco_l3_clip/us_eco_l3_clip.shp'
zonalf_path = '/content/drive/My Drive/Colab Notebooks/Analogs/Zonal/Zonal_tavg.csv'

# 2. Map Original Cluster ID to Temperature Order
raw_df = pd.read_csv(zonalf_path)
sorted_indices = raw_df['MEAN'].sort_values(ascending=False).index
temp_to_orig_map = {rank + 1: original_idx + 1 for rank, original_idx in enumerate(sorted_indices)}

# 3. Load the ISODATA clusters
with rasterio.open(iso_raster_path) as src:
    iso_arr = src.read(1)
    iso_transform = src.transform

# 4. Map the ER3 codes (US_L3CODE) to your 5 Diagnostic Groups
er3_to_macro = {
    14: 1, 81: 1,
    13: 2, 20: 2, 22: 2, 24: 2,
    5: 3, 18: 3, 19: 3, 21: 3, 80: 3,
    23: 4, 79: 4,
    25: 5, 26: 5
}

elc_gdf = gpd.read_file(elc_shapefile_path)
elc_gdf['Macro_ID'] = elc_gdf['US_L3CODE'].astype(int).map(er3_to_macro)
elc_gdf = elc_gdf.dropna(subset=['Macro_ID'])

# 5. Rasterize the ERA3 polygons to perfectly align with the ISODATA grid
elc_arr = rasterize(
    shapes=((geom, int(value)) for geom, value in zip(elc_gdf.geometry, elc_gdf['Macro_ID'])),
    out_shape=iso_arr.shape,
    transform=iso_transform,
    fill=0,
    dtype='int16'
)

# 6. Mask and flatten to only evaluate valid pixels within the study area
valid_mask = (iso_arr >= 1) & (iso_arr <= 15) & (elc_arr > 0)
iso_flat = iso_arr[valid_mask]
elc_flat = elc_arr[valid_mask]

# 7. Generate Spatial Overlap Matrix
overlap_matrix = pd.crosstab(iso_flat, elc_flat)
total_valid_pixels = len(iso_flat)

# 8. Define Manual Groups from Table 3 (Temperature Ordered)
manual_groups = {
    'Hot Desert': {'macro_id': 1, 'temp_clusters': [1, 2]},
    'Great Basin/Colorado Plateau': {'macro_id': 2, 'temp_clusters': [6, 7, 8, 9, 10, 11, 12]},
    'Northern Basin/Mountain': {'macro_id': 3, 'temp_clusters': [12, 14, 15]},
    'Mogollon/Madrean': {'macro_id': 4, 'temp_clusters': [3, 5, 13]},
    'Great Plains': {'macro_id': 5, 'temp_clusters': [4, 11, 12]}
}

# 9. Calculate Metrics Dynamically to construct Table 3
results = []
for group_name, params in manual_groups.items():
    target_macro_id = params['macro_id']
    temp_clusters = params['temp_clusters']

    orig_clusters = [temp_to_orig_map[t] for t in temp_clusters]

    if target_macro_id not in overlap_matrix.columns:
        continue

    elc_area = overlap_matrix[target_macro_id].sum()
    outside_area = total_valid_pixels - elc_area
    cluster_area = overlap_matrix.loc[overlap_matrix.index.isin(orig_clusters)].sum().sum()

    true_positives = overlap_matrix.loc[overlap_matrix.index.isin(orig_clusters), target_macro_id].sum()
    false_positives = cluster_area - true_positives
    false_negatives = elc_area - true_positives
    true_negatives = outside_area - false_positives

    tpr = true_positives / elc_area if elc_area > 0 else 0
    tnr = true_negatives / outside_area if outside_area > 0 else 0
    case_normalized_rate = (tpr + tnr) / 2
    accuracy = (true_positives + true_negatives) / total_valid_pixels

    results.append({
        'Diagnostic Group': group_name,
        'Macro-clusters': str(temp_clusters),
        'TP': true_positives,
        'TN': true_negatives,
        'FP': false_positives,
        'FN': false_negatives,
        'Prediction Accuracy': round(accuracy, 4),
        'Case-Normalized Rate': round(case_normalized_rate, 4)
    })

metrics_df = pd.DataFrame(results)

print("Table 3: ERA3 Diagnostic Group Accuracy Scores (Manual Assignments)")
print("="*95)
print(metrics_df.to_string(index=False))

Table 3: ERA3 Diagnostic Group Accuracy Scores (Manual Assignments)
            Diagnostic Group           Macro-clusters     TP      TN     FP     FN  Prediction Accuracy  Case-Normalized Rate
                  Hot Desert                   [1, 2] 162390 1420971   1212  38865               0.9753                0.9030
Great Basin/Colorado Plateau [6, 7, 8, 9, 10, 11, 12] 678312  649704  81891 213531               0.8180                0.8243
     Northern Basin/Mountain             [12, 14, 15] 144016 1195529 254723  29170               0.8251                0.8280
            Mogollon/Madrean               [3, 5, 13] 183855 1282208 131012  26363               0.9031                0.8909
                Great Plains              [4, 11, 12] 140448 1248068 228434   6488               0.8553                0.9006


In [24]:
import numpy as np
import pandas as pd
import rasterio
from rasterio.features import rasterize
import geopandas as gpd

# 1. Define Paths
iso_raster_path = '/content/drive/My Drive/Colab Notebooks/Analogs/IsoCluster.tif'
mlra_shapefile_path = '/content/drive/My Drive/Colab Notebooks/Analogs/mlra_clip/mlra_clip.shp'
zonalf_path = '/content/drive/My Drive/Colab Notebooks/Analogs/Zonal/Zonal_tavg.csv'

# 2. Map Original Cluster ID to Temperature Order
raw_df = pd.read_csv(zonalf_path)
sorted_indices = raw_df['MEAN'].sort_values(ascending=False).index
temp_to_orig_map = {rank + 1: original_idx + 1 for rank, original_idx in enumerate(sorted_indices)}

# 3. Load the ISODATA clusters
with rasterio.open(iso_raster_path) as src:
    iso_arr = src.read(1)
    iso_transform = src.transform

# 4. Map the MLRARSYM codes to the 6 Diagnostic Groups
mlra_groups = {
    1: ['30', '40'],
    2: ['24', '27', '28A', '29', '34A', '34B', '35', '42B'],
    3: ['11', '13', '22A', '23', '25', '26', '28B', '36', '46', '47', '48A', '51'],
    4: ['38', '41'],
    5: ['39'],
    6: ['42A', '42C', '70A', '70B', '77B', '77C', '77D', '77E']
}

mlra_to_macro = {code: macro_id for macro_id, codes in mlra_groups.items() for code in codes}

mlra_gdf = gpd.read_file(mlra_shapefile_path)
mlra_gdf['Macro_ID'] = mlra_gdf['MLRARSYM'].map(mlra_to_macro)
mlra_gdf = mlra_gdf.dropna(subset=['Macro_ID'])

# 5. Rasterize the MLRA polygons to perfectly align with the ISODATA grid
mlra_arr = rasterize(
    shapes=((geom, int(value)) for geom, value in zip(mlra_gdf.geometry, mlra_gdf['Macro_ID'])),
    out_shape=iso_arr.shape,
    transform=iso_transform,
    fill=0,
    dtype='int16'
)

# 6. Mask and flatten to only evaluate valid pixels within the study area
valid_mask = (iso_arr >= 1) & (iso_arr <= 15) & (mlra_arr > 0)
iso_flat = iso_arr[valid_mask]
mlra_flat = mlra_arr[valid_mask]

# 7. Generate Spatial Overlap Matrix
overlap_matrix = pd.crosstab(iso_flat, mlra_flat)
total_valid_pixels = len(iso_flat)

# 8. Define Manual Groups from Table 3 (Temperature Ordered)
manual_groups = {
    'Hot Desert': {'macro_id': 1, 'temp_clusters': [1, 2]},
    'Great Basin/Colorado Plateau': {'macro_id': 2, 'temp_clusters': [6, 7, 8, 9, 10, 11]},
    'Northern Basin/Mountain': {'macro_id': 3, 'temp_clusters': [12, 14, 15]},
    'Mogollon/Madrean': {'macro_id': 4, 'temp_clusters': [3, 5, 13]},
    'Northern Transition': {'macro_id': 5, 'temp_clusters': []}, # <-- ADD YOUR MANUAL TEMPERATURE ORDERS HERE
    'Great Plains': {'macro_id': 6, 'temp_clusters': [4, 11]}
}

# 9. Calculate Metrics Dynamically for MLRAs
results = []
for group_name, params in manual_groups.items():
    target_macro_id = params['macro_id']
    temp_clusters = params['temp_clusters']

    orig_clusters = [temp_to_orig_map[t] for t in temp_clusters]

    if target_macro_id not in overlap_matrix.columns:
        continue

    elc_area = overlap_matrix[target_macro_id].sum()
    outside_area = total_valid_pixels - elc_area
    cluster_area = overlap_matrix.loc[overlap_matrix.index.isin(orig_clusters)].sum().sum() if len(orig_clusters) > 0 else 0

    true_positives = overlap_matrix.loc[overlap_matrix.index.isin(orig_clusters), target_macro_id].sum() if len(orig_clusters) > 0 else 0
    false_positives = cluster_area - true_positives
    false_negatives = elc_area - true_positives
    true_negatives = outside_area - false_positives

    tpr = true_positives / elc_area if elc_area > 0 else 0
    tnr = true_negatives / outside_area if outside_area > 0 else 0
    case_normalized_rate = (tpr + tnr) / 2
    accuracy = (true_positives + true_negatives) / total_valid_pixels

    results.append({
        'Diagnostic Group': group_name,
        'Macro-clusters': str(temp_clusters),
        'TP': true_positives,
        'TN': true_negatives,
        'FP': false_positives,
        'FN': false_negatives,
        'Prediction Accuracy': round(accuracy, 4),
        'Case-Normalized Rate': round(case_normalized_rate, 4)
    })

metrics_df = pd.DataFrame(results)

print("Table 3 Extension: MLRA Diagnostic Group Accuracy Scores (Manual Assignments)")
print("="*95)
print(metrics_df.to_string(index=False))

Table 3 Extension: MLRA Diagnostic Group Accuracy Scores (Manual Assignments)
            Diagnostic Group       Macro-clusters     TP      TN     FP     FN  Prediction Accuracy  Case-Normalized Rate
                  Hot Desert               [1, 2] 156511 1433697   7091  26139               0.9795                0.9260
Great Basin/Colorado Plateau [6, 7, 8, 9, 10, 11] 486595  843767 109319 183757               0.8195                0.8056
     Northern Basin/Mountain         [12, 14, 15] 256370 1131593 142369  93106               0.8550                0.8109
            Mogollon/Madrean           [3, 5, 13] 132546 1295971 182321  12600               0.8799                0.8949
         Northern Transition                   []      0 1545412      0  78026               0.9519                0.5000
                Great Plains              [4, 11] 164420 1385477  40173  33368               0.9547                0.9016


In [22]:
import numpy as np
import pandas as pd
import rasterio
from rasterio.features import rasterize
import geopandas as gpd

# 1. Define Paths
iso_raster_path = '/content/drive/My Drive/Colab Notebooks/Analogs/IsoCluster.tif'
elc_shapefile_path = '/content/drive/My Drive/Colab Notebooks/Analogs/us_eco_l3_clip/us_eco_l3_clip.shp'
zonalf_path = '/content/drive/My Drive/Colab Notebooks/Analogs/Zonal/Zonal_tavg.csv'

# 2. Map Original Cluster ID to Temperature Order
raw_df = pd.read_csv(zonalf_path)
sorted_indices = raw_df['MEAN'].sort_values(ascending=False).index
temp_to_orig_map = {rank + 1: original_idx + 1 for rank, original_idx in enumerate(sorted_indices)}

# 3. Load the ISODATA clusters
with rasterio.open(iso_raster_path) as src:
    iso_arr = src.read(1)
    iso_transform = src.transform

# 4. Map the ER3 codes (US_L3CODE) to your 5 Diagnostic Groups
er3_to_macro = {
    14: 1, 81: 1,
    13: 2, 20: 2, 22: 2, 24: 2,
    5: 3, 18: 3, 19: 3, 21: 3, 80: 3,
    23: 4, 79: 4,
    25: 5, 26: 5
}

elc_gdf = gpd.read_file(elc_shapefile_path)
elc_gdf['Macro_ID'] = elc_gdf['US_L3CODE'].astype(int).map(er3_to_macro)
elc_gdf = elc_gdf.dropna(subset=['Macro_ID'])

# 5. Rasterize the ERA3 polygons to perfectly align with the ISODATA grid
elc_arr = rasterize(
    shapes=((geom, int(value)) for geom, value in zip(elc_gdf.geometry, elc_gdf['Macro_ID'])),
    out_shape=iso_arr.shape,
    transform=iso_transform,
    fill=0,
    dtype='int16'
)

# 6. Mask and flatten to only evaluate valid pixels within the study area
valid_mask = (iso_arr >= 1) & (iso_arr <= 15) & (elc_arr > 0)
iso_flat = iso_arr[valid_mask]
elc_flat = elc_arr[valid_mask]

# 7. Generate Spatial Overlap Matrix
overlap_matrix = pd.crosstab(iso_flat, elc_flat)
total_valid_pixels = len(iso_flat)

# 8. Define Strict Standard Accuracy Groups (Derived from Optimization Output)
strict_accuracy_groups = {
    'Hot Desert': {'macro_id': 1, 'temp_clusters': [1, 2]},
    'Great Basin/Colorado Plateau': {'macro_id': 2, 'temp_clusters': [6, 7, 8, 9, 10, 11, 12, 14]},
    'Northern Basin/Mountain': {'macro_id': 3, 'temp_clusters': [15]},
    'Mogollon/Madrean': {'macro_id': 4, 'temp_clusters': [5, 13]},
    'Great Plains': {'macro_id': 5, 'temp_clusters': [4]}
}

# 9. Calculate Metrics Dynamically
results = []
for group_name, params in strict_accuracy_groups.items():
    target_macro_id = params['macro_id']
    temp_clusters = params['temp_clusters']

    orig_clusters = [temp_to_orig_map[t] for t in temp_clusters]

    if target_macro_id not in overlap_matrix.columns:
        continue

    elc_area = overlap_matrix[target_macro_id].sum()
    outside_area = total_valid_pixels - elc_area
    cluster_area = overlap_matrix.loc[overlap_matrix.index.isin(orig_clusters)].sum().sum()

    true_positives = overlap_matrix.loc[overlap_matrix.index.isin(orig_clusters), target_macro_id].sum()
    false_positives = cluster_area - true_positives
    false_negatives = elc_area - true_positives
    true_negatives = outside_area - false_positives

    tpr = true_positives / elc_area if elc_area > 0 else 0
    tnr = true_negatives / outside_area if outside_area > 0 else 0
    case_normalized_rate = (tpr + tnr) / 2
    accuracy = (true_positives + true_negatives) / total_valid_pixels

    results.append({
        'Diagnostic Group': group_name,
        'Macro-clusters': str(temp_clusters),
        'TP': true_positives,
        'TN': true_negatives,
        'FP': false_positives,
        'FN': false_negatives,
        'Prediction Accuracy': round(accuracy, 4),
        'Case-Normalized Rate': round(case_normalized_rate, 4)
    })

metrics_df = pd.DataFrame(results)

print("Table 3: ERA3 Diagnostic Group Accuracy Scores (Literal Standard Accuracy Optimums)")
print("="*95)
print(metrics_df.to_string(index=False))

Table 3: ERA3 Diagnostic Group Accuracy Scores (Literal Standard Accuracy Optimums)
            Diagnostic Group               Macro-clusters     TP      TN     FP     FN  Prediction Accuracy  Case-Normalized Rate
                  Hot Desert                       [1, 2] 162390 1420971   1212  38865               0.9753                0.9030
Great Basin/Colorado Plateau [6, 7, 8, 9, 10, 11, 12, 14] 773312  592777 138818 118531               0.8415                0.8387
     Northern Basin/Mountain                         [15]  67593 1435322  14930 105593               0.9258                0.6900
            Mogollon/Madrean                      [5, 13] 137280 1367392  45828  72938               0.9268                0.8103
                Great Plains                          [4] 118642 1444828  31674  28294               0.9631                0.8930


In [13]:
import numpy as np
import pandas as pd
import rasterio
from rasterio.features import rasterize
import geopandas as gpd

# 1. Define Paths
iso_raster_path = '/content/drive/My Drive/Colab Notebooks/Analogs/IsoCluster.tif'
mlra_shapefile_path = '/content/drive/My Drive/Colab Notebooks/Analogs/mlra_clip/mlra_clip.shp'
zonalf_path = '/content/drive/My Drive/Colab Notebooks/Analogs/Zonal/Zonal_tavg.csv'

# 2. Map Original Cluster ID to Temperature Order
raw_df = pd.read_csv(zonalf_path)
sorted_indices = raw_df['MEAN'].sort_values(ascending=False).index
orig_to_temp_map = {original_idx + 1: rank + 1 for rank, original_idx in enumerate(sorted_indices)}

# 3. Load the ISODATA clusters
with rasterio.open(iso_raster_path) as src:
    iso_arr = src.read(1)
    iso_transform = src.transform

# 4. Map the MLRARSYM codes to the 6 Diagnostic Groups
mlra_groups = {
    1: ['30', '40'],                                                               # Hot Desert
    2: ['24', '27', '28A', '29', '34A', '34B', '35', '42B'],                       # Great Basin/Colorado Plateau
    3: ['11', '13', '22A', '23', '25', '26', '28B', '36', '46', '47', '48A', '51'],# Northern Basin/Mountain
    4: ['38', '41'],                                                               # Mogollon/Madrean
    5: ['39'],                                                                     # Northern Transition
    6: ['42A', '42C', '70A', '70B', '77B', '77C', '77D', '77E']                    # Great Plains
}

mlra_to_macro = {code: macro_id for macro_id, codes in mlra_groups.items() for code in codes}
macro_names = {
    1: "Hot Desert",
    2: "Great Basin/Colorado Plateau",
    3: "Northern Basin/Mountain",
    4: "Mogollon/Madrean",
    5: "Northern Transition",
    6: "Great Plains"
}

# Load MLRA shapefile and map codes
mlra_gdf = gpd.read_file(mlra_shapefile_path)
mlra_gdf['Macro_ID'] = mlra_gdf['MLRARSYM'].map(mlra_to_macro)
mlra_gdf = mlra_gdf.dropna(subset=['Macro_ID'])

# 5. Rasterize the MLRA polygons to perfectly align with the ISODATA grid
mlra_arr = rasterize(
    shapes=((geom, int(value)) for geom, value in zip(mlra_gdf.geometry, mlra_gdf['Macro_ID'])),
    out_shape=iso_arr.shape,
    transform=iso_transform,
    fill=0,
    dtype='int16'
)

# 6. Mask and flatten to only evaluate valid pixels within the study area
valid_mask = (iso_arr >= 1) & (iso_arr <= 15) & (mlra_arr > 0)
iso_flat = iso_arr[valid_mask]
mlra_flat = mlra_arr[valid_mask]

# 7. Generate Spatial Overlap Matrix and Pre-calculate areas
overlap_matrix = pd.crosstab(iso_flat, mlra_flat)
total_valid_pixels = len(iso_flat)
cluster_areas = pd.Series(iso_flat).value_counts()
mlra_areas = pd.Series(mlra_flat).value_counts()

# 8. Optimize for Literal Standard Accuracy & Calculate Metrics Dynamically
results = []
for macro_id, group_name in macro_names.items():
    if macro_id not in overlap_matrix.columns:
        continue

    elc_total_area = mlra_areas[macro_id]
    outside_total_area = total_valid_pixels - elc_total_area

    assigned_for_accuracy = []

    # --- Strict Standard Accuracy Optimization Loop ---
    for cluster_id in overlap_matrix.index:
        overlap_area = overlap_matrix.loc[cluster_id, macro_id] if macro_id in overlap_matrix.columns else 0
        cluster_total_area = cluster_areas[cluster_id]
        outside_spill = cluster_total_area - overlap_area

        # Assign ONLY if True Positives gained > True Negatives lost
        if overlap_area > outside_spill:
            assigned_for_accuracy.append(cluster_id)

    # Calculate final metrics for the optimized MLRA group
    temp_clusters = sorted([orig_to_temp_map[c] for c in assigned_for_accuracy])

    elc_area = elc_total_area
    outside_area = outside_total_area
    cluster_area = cluster_areas[cluster_areas.index.isin(assigned_for_accuracy)].sum() if len(assigned_for_accuracy) > 0 else 0

    true_positives = overlap_matrix.loc[overlap_matrix.index.isin(assigned_for_accuracy), macro_id].sum() if len(assigned_for_accuracy) > 0 else 0
    false_positives = cluster_area - true_positives
    false_negatives = elc_area - true_positives
    true_negatives = outside_area - false_positives

    tpr = true_positives / elc_area if elc_area > 0 else 0
    tnr = true_negatives / outside_area if outside_area > 0 else 0
    case_normalized_rate = (tpr + tnr) / 2
    accuracy = (true_positives + true_negatives) / total_valid_pixels

    results.append({
        'Diagnostic Group': group_name,
        'Macro-clusters': str(temp_clusters),
        'TP': true_positives,
        'TN': true_negatives,
        'FP': false_positives,
        'FN': false_negatives,
        'Prediction Accuracy': round(accuracy, 4),
        'Case-Normalized Rate': round(case_normalized_rate, 4)
    })

# Output the final MLRA metrics
metrics_df = pd.DataFrame(results)

print("Table 3 Extension: MLRA Diagnostic Group Accuracy Scores (Literal Standard Accuracy Optimums)")
print("="*95)
print(metrics_df.to_string(index=False))

Table 3 Extension: MLRA Diagnostic Group Accuracy Scores (Literal Standard Accuracy Optimums)
            Diagnostic Group       Macro-clusters     TP      TN     FP     FN  Prediction Accuracy  Case-Normalized Rate
                  Hot Desert               [1, 2] 156511 1433697   7091  26139               0.9795                0.9260
Great Basin/Colorado Plateau [6, 7, 8, 9, 10, 11] 486595  843767 109319 183757               0.8195                0.8056
     Northern Basin/Mountain             [14, 15] 184476 1223988  49974 165000               0.8676                0.7443
            Mogollon/Madrean               [3, 5] 122900 1366503 111789  22246               0.9174                0.8856
         Northern Transition                   []      0 1545412      0  78026               0.9519                0.5000
                Great Plains                  [4] 147805 1423139   2511  49983               0.9677                0.8728
